In [3]:
import pandas as pd
import re
import numpy as np

In [4]:
with open ('/home/tjroot/Bioinformatics/Research/FlyntLab/AutoDeep/Arabidopsis/mirdeep/output.mrd') as f:
    read_data = f.read().splitlines()

In [5]:
read_data

['>NC_003076.8_40575',
 'score total\t\t      295567.5',
 'score for star read(s)\t           3.9',
 'score for read counts\t      295557.5',
 'score for mfe\t\t           1.5',
 'score for randfold\t           1.6',
 'score for cons. seed\t             3',
 'miRNA with same seed\t aly-miR165a-3p',
 'total read count\t        579735',
 'mature read count\t        579656',
 'loop read count\t\t             3',
 'star read count\t\t            76',
 'exp                           ffSSSSSSSSSSSSSSSSSSSSSlllllllllllllllllllllllllllllllllllllllllllllllMMMMMMMMMMMMMMMMMMMMMfffffffffffffffffffff',
 'obs                           ffSSSSSSSSSSSSSSSSSSSSSlllllllllllllllllllllllllllllllllllllllllllllllMMMMMMMMMMMMMMMMMMMMMfffffffffffffffffffff',
 'pri_seq                       ggugaaugaugccuggcucgagaccauucaaucucaugaucucaugauuauaacgaugaugaugaugaugucggaccaggcuucauuccccucaacuuacacguuuugcuuc',
 'pri_struct                    ((.((((((.((((((..(((.(.((((..(((((((((((....))))))...)).)))...)))).).)))..)

In [6]:
entries = []
temp = []

for item in read_data:
    if item.startswith('>'):
        if temp:
            entries.append(temp)
        temp = [item]
    else:
        if not item == '':
            temp.append(item)

if temp:
    entries.append(temp)



In [7]:
entries[1]

['>NC_003076.8_41625',
 'score total\t\t      295745.8',
 'score for star read(s)\t           3.9',
 'score for read counts\t      295735.9',
 'score for mfe\t\t           1.4',
 'score for randfold\t           1.6',
 'score for cons. seed\t             3',
 'miRNA with same seed\t aly-miR165a-3p',
 'total read count\t        580085',
 'mature read count\t        580058',
 'loop read count\t\t 0',
 'star read count\t\t            27',
 'exp                           fffSSSSSSSSSSSSSSSSSSSSSllllllllllllllllllllllllllllllllllllllllllllllMMMMMMMMMMMMMMMMMMMMMfffffffffffffffffffff',
 'obs                           fffSSSSSSSSSSSSSSSSSSSSSllllllllllllllllllllllllllllllllllllllllllllllMMMMMMMMMMMMMMMMMMMMMfffffffffffffffffffff',
 'pri_seq                       agaggaauguuguuuggcucgaggucauggagaguaauucguuaacccaacucaaaacucuaaaugauucucggaccaggcuucauuccccucaaccuauuuuaucgcauuu',
 'pri_struct                    ((.((((((..((((((..(((((((((..(((((.....(((.....))).....)))))..)))).)))))..))))))..)))))

In [8]:
# cell 7 – rewritten
import re
import pandas as pd

def parse_seq_entry(seq_line):
    s = seq_line.strip()
    # drop trailing tab/flag like "\t0" if present
    s = re.sub(r'\t.*$', '', s).strip()
    # remove surrounding quotes if any
    s = s.strip('\'"')
    m = re.match(r'^(seq_[0-9]+)_x(\d+)\s+(.*)$', s)
    consensus = re.match(r'(exp|obs)\s(.+)$', s)

    if consensus:
        return 'consensus', 1, consensus.group(2).strip()
    if m:
        return m.group(1), int(m.group(2)), m.group(3).strip()
    # fallback: first token contains id_count, rest is alignment
    parts = s.split(None, 1)
    if parts:
        m2 = re.match(r'^(seq_[0-9]+)_x(\d+)$', parts[0])
        if m2:
            alignment = parts[1].strip() if len(parts) > 1 else ''
            return m2.group(1), int(m2.group(2)), alignment
    return None, None, s

# build dataframes, dropping the 'exp' consensus only when an 'obs' is present
dfs = []
for idx, entry in enumerate(entries):
    has_obs = any(line.strip().startswith('obs') for line in entry)
    rows = []
    miRNA_id = ''
    for line in entry:
        if line.startswith('>'):
            miRNA_id = line[1:].strip()  # store miRNA id for this entry
        if has_obs and line.strip().startswith('exp'):
            # skip the exp consensus if an obs consensus exists in this entry for more accurate downstream analysis
            continue
        name, count, aln = parse_seq_entry(line)
        if name is not None:
            rows.append((name, count, aln))

    df = pd.DataFrame(rows, columns=['sequence_name', 'sequence_count', 'alignment'])
    dfs.append((df, miRNA_id))

# example
dfs[1][0].head()

,sequence_name,sequence_count,alignment
0,consensus,1,fffSSSSSSSSSSSSSSSSSSSSSllllllllllllllllllllll...
1,seq_90106197,1,...ggaauguuguuuggcAcgagg.........................
2,seq_85345768,2,...ggaCuguuguuuggcucgagg.........................
3,seq_78941926,16,...ggaauguuguCuggcucgagg.........................
4,seq_87544131,1,...ggaauguuguCuggcucgaggu........................


In [9]:
combined = pd.concat(
    [df.assign(entry_index=i, miRNA_id=miRNA_id) for i, (df, miRNA_id) in enumerate(dfs)],
    ignore_index=True
)
combined.head()

,sequence_name,sequence_count,alignment,entry_index,miRNA_id
0,consensus,1,ffSSSSSSSSSSSSSSSSSSSSSlllllllllllllllllllllll...,0,NC_003076.8_40575
1,seq_78573590,19,..ugaaugaugccuggcucg.............................,0,NC_003076.8_40575
2,seq_83466997,3,..ugaaugaugccuggcucgag...........................,0,NC_003076.8_40575
3,seq_76226770,48,..ugaaugaugccuggcucgaga..........................,0,NC_003076.8_40575
4,seq_86930278,1,..uNaaugaugccuggcucgaga..........................,0,NC_003076.8_40575


In [10]:
test_case = combined[combined['entry_index'] == 2]
test_case

,sequence_name,sequence_count,alignment,entry_index,miRNA_id
689,consensus,1,ffffffffffffffffffffMMMMMMMMMMMMMMMMMMMMMlllll...,2,NC_003070.9_478
690,seq_83987281,3,..uuaauggcuucacucuuc.............................,2,NC_003070.9_478
691,seq_87828238,1,...............ucuucuuuggauugaagggagcucuu........,2,NC_003070.9_478
692,seq_90441677,1,................cuucuuuggauugaagggagU............,2,NC_003070.9_478
693,seq_86009188,2,..................ucuuuggauugaagggagc............,2,NC_003070.9_478
...,...,...,...,...,...
1110,seq_90400386,1,......................uggauugaagggagGucuu........,2,NC_003070.9_478
1111,seq_81636729,6,......................uggauugaagggagcucuu........,2,NC_003070.9_478
1112,seq_87931423,1,......................Aggauugaagggagcucuu........,2,NC_003070.9_478
1113,seq_88153969,1,......................uggauugaagggagcucuuc.......,2,NC_003070.9_478


Extract 5' start coordinate: For each read alignment string, find the index of the first nucleotide character (A/C/G/T/U/N, case-insensitive). Use the read's _x<count> as weight. Optionally convert index → genomic/pre-miRNA coordinate if you have a reference offset.

Dominant-start proportion:

Compute counts p_i for each 5' start position i (weighted by _x<count>).
Metric: dominant_prop = max(p_i) / sum(p_i).
Interpretation: high (e.g., >0.7–0.8) → strong 5' homogeneity.
Normalized Shannon entropy:

H = -sum_i p_i * log(p_i). Normalized H' = H / log(k) where k = number of observed start positions.
Low H' (close to 0) → homogeneous; high (close to 1) → heterogeneous. Thresholds: H' < 0.25 strong, 0.25–0.5 moderate.
Gini coefficient / inequality:

Compute Gini on the start counts distribution. Higher Gini → more concentrated (homogeneous). Use as alternative to entropy.
n90 (compactness):

Compute the minimum number of distinct start positions that together account for 90% of reads. n90 ≤ 2 → homogeneous.
Tolerance-window aggregation (handles micro-heterogeneity ±1 nt):

Merge counts across windows of ±1 (or ±2) nt and recompute dominant_prop / H'. Useful because enzymatic processing can produce 1-nt shifts.
Peak detection / KDE:

Smooth start-position counts with a small kernel, identify peak(s), compute peak height vs background (peak / median or peak / mean). High peak/background indicates homogeneity.
Statistical test vs null:

Null: uniform or background distribution from reads mapped elsewhere. Use chi-square or permutation (shuffle read starts among positions) to obtain p-value that observed concentration is non-random.
Sequence-consistency at 5' nucleotide:

For reads starting at the dominant position, compute fraction with identical first nucleotide and identical first 2–3 nt. True miRNAs often show consistent 5' bases.
Composite homogeneity score:

Combine metrics, e.g.: score = 0.5dominant_prop + 0.3(1 - H') + 0.2*(1 - n90/10) (normalize appropriately). Tune weights on labeled data.
Extraction pseudocode (5' pos + counts):

For each line in alignments:
parse count from name _x\d+
extract aln = alignment string (strip trailing \t flags)
start_idx = index of first nucleotide character in aln (0-based)
increment position counter: counts[start_idx] += count
Suggested thresholds (starting points):

dominant_prop ≥ 0.7 and normalized entropy ≤ 0.3 and n90 ≤ 2 → call highly_homogeneous.
dominant_prop 0.5–0.7 → moderately_homogeneous.
else → heterogeneous.
Notes / caveats:

Decide coordinate frame (alignment index vs genome coordinate). If alignments include flanking dots, using first-nucleotide index is sufficient for comparing within an entry.
Use _x<count> weights to avoid treating each distinct sequence string equally when counts vary widely.
Allow ±1 nt tolerance for micro-heterogeneity — recompute metrics with and without tolerance to report both strict and tolerant homogeneity.
Next step: implement one or more of these metrics in the notebook and add summary columns (dominant_prop, entropy, n90, composite_score) to your combined DataFrame. Want me to implement the strict + ±1-nt tolerant versions as notebook cells and show the results?

GPT-5 mini • 1x

In [11]:
test_alignments, test_counts = test_case['alignment'].to_numpy(), test_case['sequence_count'].to_numpy()
test_alignments

test_case.reset_index().loc[0,'alignment']
test_case

,sequence_name,sequence_count,alignment,entry_index,miRNA_id
689,consensus,1,ffffffffffffffffffffMMMMMMMMMMMMMMMMMMMMMlllll...,2,NC_003070.9_478
690,seq_83987281,3,..uuaauggcuucacucuuc.............................,2,NC_003070.9_478
691,seq_87828238,1,...............ucuucuuuggauugaagggagcucuu........,2,NC_003070.9_478
692,seq_90441677,1,................cuucuuuggauugaagggagU............,2,NC_003070.9_478
693,seq_86009188,2,..................ucuuuggauugaagggagc............,2,NC_003070.9_478
...,...,...,...,...,...
1110,seq_90400386,1,......................uggauugaagggagGucuu........,2,NC_003070.9_478
1111,seq_81636729,6,......................uggauugaagggagcucuu........,2,NC_003070.9_478
1112,seq_87931423,1,......................Aggauugaagggagcucuu........,2,NC_003070.9_478
1113,seq_88153969,1,......................uggauugaagggagcucuuc.......,2,NC_003070.9_478


In [ ]:
# Extract 5' starting nucleotide (first nucleotide character) and its index for each alignment
five_prime_data = []

for i, aln in enumerate(test_alignments[1:]):  # skip consensus
    # Find index of first nucleotide (A, C, G, T, U, N - case insensitive)
    nucleotide_pattern = r'[ACGTUNacgtun]'
    match = re.search(nucleotide_pattern, aln)
    
    if match:
        idx = match.start()
        nucleotide = match.group().upper()
    else:
        # If no nucleotide found, record -1 and 'N'
        idx = -1
        nucleotide = 'N'
    
    # Get count from test_counts
    count = test_counts[i]
    location = test_alignments[0][idx] if idx != -1 and idx < len(test_alignments[0]) else 'N/A'
    size = test_alignments[0].count(location) if location != 'N/A' else 0
    five_prime_data.append({
        'five_prime_index': idx,
        'five_prime_nucleotide': nucleotide,
        'sequence_count': count,
        'alignment_location' : location,
        'mature_size': size
    })

# Create DataFrame
five_prime_df = pd.DataFrame(five_prime_data)
five_prime_df

,five_prime_index,five_prime_nucleotide,sequence_count,alignment_location,mature_size
0,2,U,1,f,40
1,15,U,3,f,40
2,16,C,1,f,40
3,18,U,1,f,40
4,18,U,2,f,40
...,...,...,...,...,...
420,22,U,1,M,21
421,22,U,1,M,21
422,22,A,6,M,21
423,22,U,1,M,21


In [55]:
# from scipy import stats
# import matplotlib.pyplot as plt

# # Weight five_prime_indices by sequence_counts
# weighted_indices = five_prime_df.loc[five_prime_df.index.repeat(five_prime_df['sequence_count']),'five_prime_index'].values

# # Create KDE plot with weighted indices
# kde = stats.gaussian_kde(weighted_indices)
# x_range = np.linspace(weighted_indices.min(), weighted_indices.max(), 200)

# plt.figure(figsize=(12, 5))
# plt.plot(x_range, kde(x_range), linewidth=2)
# plt.fill_between(x_range, kde(x_range), alpha=0.3)
# plt.xlabel('5\' Start Index')
# plt.ylabel('Density')
# plt.title('KDE Plot of 5\' Start Positions (Weighted by Sequence Count)')
# plt.grid(True, alpha=0.3)
# plt.show()

In [56]:
five_prime_df['sequence_count'].describe()

count       425.000000
mean       1783.174118
std       24784.342844
min           1.000000
25%           2.000000
50%           9.000000
75%          49.000000
max      485424.000000
Name: sequence_count, dtype: float64

In [57]:
M_idx = five_prime_df.groupby('five_prime_index')['sequence_count'].sum().sort_values(ascending=False).idxmax()
five_prime_df.groupby('five_prime_index')['sequence_count'].sum().sort_values(ascending=False)

five_prime_index
20    754478
21      2327
19       939
18        58
22        42
15         3
2          1
16         1
Name: sequence_count, dtype: int64

In [1]:
five_prime_df

NameError: name 'five_prime_df' is not defined

In [59]:
def is_homogeneous(five_prime_index):
    """
    Improved homogeneity metrics for a given 5' start position.
    
    Returns:
    - max_prop: Proportion of reads at single most abundant position
    - top1/2/3_prop: Cumulative proportion in top N positions
    - normalized_entropy: Shannon entropy normalized by max entropy
    - gini_coefficient: Inequality measure (0=uniform, 1=concentrated)
    - n_distinct_positions: Number of different 5' start positions
    - coefficient_variation: Relative spread of counts
    - total_count: Total reads in window
    """
    # Determine the mature miRNA size range (typically 20-24 nt)
    alignment_position = five_prime_df.loc[five_prime_df['five_prime_index'] == five_prime_index, 'alignment_location'].iloc[0]
    if alignment_position == 'M' or alignment_position == 'S':
        miRNA_size = test_alignments[0].count(alignment_position)
    else:
        miRNA_size = 22  # default to 22nt if alignment location is not M or S
    
    
    min_idx = five_prime_index - miRNA_size
    max_idx = five_prime_index + miRNA_size
    
    # Filter reads within the tolerance window
    filtered = five_prime_df[
        (five_prime_df['five_prime_index'] >= min_idx) & 
        (five_prime_df['five_prime_index'] <= max_idx)
    ]
    if filtered.empty:
        return None
    
    # AGGREGATION BY POSITION: Group sequences by 5' start position and sum counts
    position_counts = filtered.groupby('five_prime_index')['sequence_count'].sum().values
    total = position_counts.sum()
    proportions = position_counts / total
    
    # Metric 1: Maximum proportion (reads at most abundant position)
    max_prop = proportions.max()
    # Find positions achieving max_prop
    position_agg = filtered.groupby('five_prime_index')['sequence_count'].sum().sort_values(ascending=False)
    max_position_indices = position_agg[position_agg == max_prop].index
    print("\nMAX_PROP positions in filtered window:")
    print(filtered[filtered['five_prime_index'].isin(max_position_indices)])
    
    # determine dominant nucleotide and position among filtered
    dominant_position = position_agg.index[0]
    dominant_nucleotide = filtered[filtered['five_prime_index'] == dominant_position]['five_prime_nucleotide'].iloc[0]
    
    # restrict to rows with dominant nucleotide
    nucleotide_filtered = filtered[filtered['five_prime_nucleotide'] == dominant_nucleotide]
    if nucleotide_filtered.empty:
        nucleotide_filtered = filtered
    
    # further restrict to exact five_prime_index for top-N metrics
    pos_filtered = nucleotide_filtered[nucleotide_filtered['five_prime_index'] == five_prime_index]
    use_filtered = pos_filtered if not pos_filtered.empty else nucleotide_filtered
    if pos_filtered.empty:
        print(f"WARNING: no reads exactly at position {five_prime_index}; using {len(use_filtered)} reads for top-N metrics")
    
    # compute proportions over positions (aggregate by position)
    sub_position_agg = use_filtered.groupby('five_prime_index')['sequence_count'].sum().sort_values(ascending=False)
    sub_position_counts = sub_position_agg.values
    sub_total = sub_position_counts.sum()
    sub_props = sub_position_counts / sub_total if sub_total > 0 else sub_position_counts
    
    # sort by these proportions (already sorted from aggregation)
    sorted_idx = np.arange(len(sub_props))
    sorted_props = sub_props
    
    print(f"\n{'='*80}")
    print(f"5' Index: {five_prime_index} | Dominant Nucleotide: {dominant_nucleotide}")
    print(f"Rows with exact position: {len(pos_filtered)}; using {len(use_filtered)} rows for top-N")
    print(f"{'='*80}\n")
    
    # TOP 1
    top1_position = sub_position_agg.index[0]
    print("TOP1 position(s):")
    print(use_filtered[use_filtered['five_prime_index'] == top1_position])
    top1_prop = sorted_props[0] if len(sorted_props) > 0 else 0
    print(f"TOP1 proportion: {top1_prop:.4f}\n")
    
    # TOP 2
    count_n = min(2, len(sorted_props))
    top2_positions = sub_position_agg.index[:count_n]
    print("TOP2 position(s):")
    print(use_filtered[use_filtered['five_prime_index'].isin(top2_positions)])
    top2_prop = sorted_props[:count_n].sum() if len(sorted_props) >= count_n else sorted_props.sum()
    print(f"TOP2 proportion: {top2_prop:.4f}\n")
    
    # TOP 3
    count_n = min(3, len(sorted_props))
    top3_positions = sub_position_agg.index[:count_n]
    print("TOP3 position(s):")
    print(use_filtered[use_filtered['five_prime_index'].isin(top3_positions)])
    top3_prop = sorted_props[:count_n].sum() if len(sorted_props) >= count_n else sorted_props.sum()
    print(f"TOP3 proportion: {top3_prop:.4f}\n")
    
    # Metric 3: Normalized Shannon entropy (over aggregated position counts)
    entropy = -np.sum(proportions * np.log2(proportions + 1e-10))
    n_distinct_positions = len(position_counts)
    max_entropy = np.log2(n_distinct_positions) if n_distinct_positions > 0 else 0
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    
    # Metric 4: Gini coefficient (0=uniform, 1=concentrated) over positions
    sorted_position_counts = np.sort(position_counts)
    n = len(sorted_position_counts)
    gini = (2 * np.sum(np.arange(1, n+1) * sorted_position_counts)) / (n * sorted_position_counts.sum()) - (n + 1) / n
    
    # Metric 6: Coefficient of variation of position counts
    cv = np.std(position_counts) / np.mean(position_counts) if np.mean(position_counts) > 0 else 0
    
    return {
        'star_or_mature' : alignment_position,
        'five_prime_index': five_prime_index,
        'max_prop': max_prop,
        'top1_prop': top1_prop,
        'top2_prop': top2_prop,
        'top3_prop': top3_prop,
        'normalized_entropy': normalized_entropy,
        'gini_coefficient': gini,
        'n_distinct_positions': n_distinct_positions,
        'coefficient_variation': cv,
        'total_count': total
    }


In [60]:
is_homogeneous(M_idx)


MAX_PROP positions in filtered window:
Empty DataFrame
Columns: [five_prime_index, five_prime_nucleotide, sequence_count, alignment_location]
Index: []

5' Index: 20 | Dominant Nucleotide: U
Rows with exact position: 306; using 306 rows for top-N

TOP1 position(s):
     five_prime_index five_prime_nucleotide  sequence_count alignment_location
49                 20                     U               1                  M
50                 20                     U               5                  M
51                 20                     U              18                  M
52                 20                     U               3                  M
53                 20                     U               1                  M
..                ...                   ...             ...                ...
369                20                     U               4                  M
370                20                     U               1                  M
371                20 

{'star_or_mature': 'M',
 'five_prime_index': 20,
 'max_prop': 0.995551884346354,
 'top1_prop': 1.0,
 'top2_prop': 1.0,
 'top3_prop': 1.0,
 'normalized_entropy': 0.015317028701467629,
 'gini_coefficient': 0.8734907943402974,
 'n_distinct_positions': 8,
 'coefficient_variation': 2.632313851722562,
 'total_count': 757849}

In [ ]:
#max_position = selection_df.iloc[np.argmax(selection_df['Counts']),:]['Position']

# 					selected_rows = selection_df[(selection_df['Leftmost Nucleotide'] == max_leftmost_nuc) & (selection_df['Position'] == max_position)]



					
# 					if num_pages > selected_rows.shape[0]:
# 						num_pages = selected_rows.shape[0]
					
# 					top_loci = selected_rows.iloc[np.argpartition(selected_rows['Counts'], -num_pages)[-num_pages:],:]
# 					tdf = selection_df[(selection_df['Position'] - max_position) <= 20]


# 					loci_count = selection_df.shape[0]


# 					five_prime_process_signal_selected_row_count_sum_over_mature_count_sum.append(sum(selected_rows['Counts'])/sum(tdf['Counts']))